In [45]:
#Import packages
import pandas as pd 
import openpyxl as opxl

from database import load_data

from database import get_connection, load_data



In [5]:
df_brand = pd.read_excel(r"C:\Users\khwk503\AZCollaboration\AZVN ComEx - AZVN Dashboard\Pre_CP_Monthly_Dashboard\CategoryBank.xlsx", sheet_name="Brand")
df_channel = pd.read_excel(r"C:\Users\khwk503\AZCollaboration\AZVN ComEx - AZVN Dashboard\Pre_CP_Monthly_Dashboard\CategoryBank.xlsx", sheet_name="Channel")
df_region = pd.read_excel(r"C:\Users\khwk503\AZCollaboration\AZVN ComEx - AZVN Dashboard\Pre_CP_Monthly_Dashboard\CategoryBank.xlsx", sheet_name="Region")
df_teamstructure = pd.read_excel(r"C:\Users\khwk503\AZCollaboration\AZVN ComEx - AZVN Dashboard\Pre_CP_Monthly_Dashboard\CategoryBank.xlsx", sheet_name="Teamstructure")

In [13]:
df_channel.head(10)

,ChannelD,CustomerTypeName,Group Channel 1,Group Channel 2,Group Channel 3,Group Channel 4,Sort,SortChannel4,Group Channel 5,SortChannel5
0,101,21-Gov. Hosp - Dept,Hospital Insurance,Hospital,INS-Hos Gov,Gov Hos,1,1,Tender,1
1,102,22-Gov. Hosp - Phar,Hospital Selfpaid,Hospital,OOP-Hos Gov,Gov Hos,2,1,Non tender,2
2,103,25-Polyclinic,Hospital Selfpaid,Clinic,OOP-Poly,Pri Hos,5,2,Non tender,2
3,104,23-Private Hosp - Dept,Hospital Selfpaid,Hospital,OOP-Hos Pri,Pri Hos,3,2,Non tender,2
4,105,26-Pharmacy,Retail,Retail,OOP-Ind.Phar,Non Chain,8,4,Non chain,3
5,106,24-Private Hosp - Phar,Hospital Selfpaid,Hospital,OOP-Hos Pri,Pri Hos,4,2,Non tender,2
6,107,27-Wholesalers,Retail,Retail,OOP-WS,Non Chain,9,4,Non chain,3
7,108,28-Monoclinic,Hospital Selfpaid,Clinic,OOP-Mono,Non Chain,6,4,Non tender,2
8,109,29-Pharmacy Chain,Retail,Retail,OOP-PMC,Chain,7,3,Chain,4


In [1]:
import pyodbc
import pandas as pd
 
server = r"SGKCAWCUSDBDV02.astrazeneca.net"        
database = "VNDB_O365"
username = "VNDEV"
password = "vnhc-21"
 
conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};DATABASE={database};UID={username};PWD={password};"
)
 
with pyodbc.connect(conn_str) as conn:
    df_customerdetail = pd.read_sql(
        """
        SELECT T0.CustomerID, T0.CustomerName, T0.Address, T0.RegionID,
               T0.CityID, T0.AccountID, T1.AccountName,T1.AccountDisplay,
               CASE
                   WHEN T0.ChannelName = '21-Gov. Hosp - Dept'    THEN '101'
                   WHEN T0.ChannelName = '22-Gov. Hosp - Phar'    THEN '102'
                   WHEN T0.ChannelName = '23-Private Hosp - Dept' THEN '104'
                   WHEN T0.ChannelName = '24-Private Hosp - Phar' THEN '106'
                   WHEN T0.ChannelName = '25-Polyclinic'          THEN '103'
                   WHEN T0.ChannelName = '26-Pharmacy'            THEN '105'
                   WHEN T0.ChannelName = '27-Wholesalers'         THEN '107'
                   WHEN T0.ChannelName = '28-Monoclinic'          THEN '108'
                   WHEN T0.ChannelName = '29-Pharmacy Chain'      THEN '109'
               END AS ChannelID
        FROM [VNDB_O365].[dbo].[vw_Sipro_Customer] AS T0
        LEFT JOIN [VNDB_O365].[dbo].[vw_ParentAccount] T1
            ON T0.CUSTOMERID = T1.CUSTOMERID
        """,
        conn
    )

    df_monthlysales = pd.read_sql(
        """
                SELECT T0.CustomerID, T0.Qty, T1.TerritoryID, T0.Cust_Brand,
               T0.BrandID, T0.ProductID, T0.ByMonth, T0.ByYear,
               T0.YYMM, T0.AmountUSD, T1.ProductWeight
        FROM (
            SELECT CustomerID, AmountUSD, Qty, BrandID,
                   ProductID, ByMonth, ByYear,
                   YYMM, CONCAT(CustomerID, Brand) AS Cust_Brand
            FROM [VNDB_O365].[dbo].[vw_SIpro_CustomerProductByCutoff]
        ) AS T0
        LEFT JOIN (
            SELECT TeamID, TerritoryID, CustomerID, BrandID,
                   ProductWeight, Brand,
                   CONCAT(CustomerID, Brand) AS Cust_Brand
            FROM [VNDB_O365].[dbo].[vw_SIpro_TerritoryAlignment]
        ) AS T1 ON T0.Cust_Brand = T1.Cust_Brand
    """, conn)


C:\Users\khwk503\AppData\Local\Temp\ipykernel_23272\3431229778.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customerdetail = pd.read_sql(
C:\Users\khwk503\AppData\Local\Temp\ipykernel_23272\3431229778.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_monthlysales = pd.read_sql(


In [ ]:
df = pd.merge(df_monthlysales, df_customerdetail, on="CustomerID", how="left")

In [8]:
len(df)

708811

In [10]:
df_brand['BrandID'] = df_brand['BrandID'].astype(str)
df = pd.merge(df, df_brand[["BrandID", "Brand"]], on="BrandID", how="left")

In [29]:
df.head(5)

,CustomerID,Qty,TerritoryID,Cust_Brand,BrandID,ProductID,ByMonth,ByYear,YYMM,AmountUSD,ProductWeight,CustomerName,Address,RegionID,CityID,AccountID,AccountName,AccountDisplay,ChannelID,Brand
0,30172230,2.0,655.0,30172230NEXIUM SACHET,10205,10205101,11,2024,202411,45.0239,1.0,NHA THUOC QUANG TRUNG,7 QUANG TRUNG- P. HOI THUONG,202,124,CA001959,NHA THUOC QUANG TRUNG,NT QUANG TRUNG,105,NEXIUM SACHET
1,30172230,2.0,NaN,30172230BRILINTA,10313,10313101,1,2022,202201,75.1172,NaN,NHA THUOC QUANG TRUNG,7 QUANG TRUNG- P. HOI THUONG,202,124,CA001959,NHA THUOC QUANG TRUNG,NT QUANG TRUNG,105,BRILINTA
2,30172230,2.0,NaN,30172230BRILINTA,10313,10313101,9,2022,202209,75.1172,NaN,NHA THUOC QUANG TRUNG,7 QUANG TRUNG- P. HOI THUONG,202,124,CA001959,NHA THUOC QUANG TRUNG,NT QUANG TRUNG,105,BRILINTA
3,30172230,1.0,NaN,30172230BRILINTA,10313,10313101,2,2023,202302,30.0469,NaN,NHA THUOC QUANG TRUNG,7 QUANG TRUNG- P. HOI THUONG,202,124,CA001959,NHA THUOC QUANG TRUNG,NT QUANG TRUNG,105,BRILINTA
4,30172230,0.0,755.0,30172230FORXIGA,10903,10903102,9,2022,202209,0.0000,1.0,NHA THUOC QUANG TRUNG,7 QUANG TRUNG- P. HOI THUONG,202,124,CA001959,NHA THUOC QUANG TRUNG,NT QUANG TRUNG,105,FORXIGA


In [ ]:
df_channel = df_channel.rename(columns={"ChannelD": "ChannelID"})
df_channel['ChannelID'] = df_channel['ChannelID'].astype(str)
df = pd.merge(df, df_channel[["ChannelID", "Group Channel 1"]], on="ChannelID", how="left")

ValueError: You are trying to merge on object and int64 columns for key 'ChannelID'. If you wish to proceed you should use pd.concat

In [28]:
df_channel.head(10)

,ChannelID,CustomerTypeName,Group Channel 1,Group Channel 2,Group Channel 3,Group Channel 4,Sort,SortChannel4,Group Channel 5,SortChannel5
0,101,21-Gov. Hosp - Dept,Hospital Insurance,Hospital,INS-Hos Gov,Gov Hos,1,1,Tender,1
1,102,22-Gov. Hosp - Phar,Hospital Selfpaid,Hospital,OOP-Hos Gov,Gov Hos,2,1,Non tender,2
2,103,25-Polyclinic,Hospital Selfpaid,Clinic,OOP-Poly,Pri Hos,5,2,Non tender,2
3,104,23-Private Hosp - Dept,Hospital Selfpaid,Hospital,OOP-Hos Pri,Pri Hos,3,2,Non tender,2
4,105,26-Pharmacy,Retail,Retail,OOP-Ind.Phar,Non Chain,8,4,Non chain,3
5,106,24-Private Hosp - Phar,Hospital Selfpaid,Hospital,OOP-Hos Pri,Pri Hos,4,2,Non tender,2
6,107,27-Wholesalers,Retail,Retail,OOP-WS,Non Chain,9,4,Non chain,3
7,108,28-Monoclinic,Hospital Selfpaid,Clinic,OOP-Mono,Non Chain,6,4,Non tender,2
8,109,29-Pharmacy Chain,Retail,Retail,OOP-PMC,Chain,7,3,Chain,4
